In [81]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
# t_res='5min'; res='1km'
# res='1km'
# Np_str='1e6'

# dx = 1km; Np = 50M
#Importing Model Data
check=False
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'


# # dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***
# res='250m'
# Np_str='150e6'

In [82]:
#JOB ARRAY SETUP
job_array=True
if job_array==True:

    num_jobs=360 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***
    total_elements=len(data['time']) #total num of variables

    if num_jobs >= total_elements:
        raise ValueError("Number of jobs cannot be greater than or equal to total elements.")
    
    job_range = total_elements // num_jobs  # Base size for each chunk
    remaining = total_elements % num_jobs   # Number of chunks with 1 extra 
    
    # Function to compute the start and end for each job_id
    def get_job_range(job_id, num_jobs):
        job_id-=1
        # Add one extra element to the first 'remaining' chunks
        start_job = job_id * job_range + min(job_id, remaining)
        end_job = start_job + job_range + (1 if job_id < remaining else 0)
    
        if job_id == num_jobs - 1: 
            end_job = total_elements #- 1
        return start_job, end_job
    # def job_testing():
    #     #TESTING
    #     start=[];end=[]
    #     for job_id in range(1,num_jobs+1):
    #         start_job, end_job = get_job_range(job_id)
    #         print(start_job,end_job)
    #         start.append(start_job)
    #         end.append(end_job)
    #     print(np.all(start!=end))
    #     print(len(np.unique(start))==len(start))
    #     print(len(np.unique(end))==len(end))
    # job_testing()
    
    job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
    if job_id==0: job_id=10
    start_job, end_job = get_job_range(job_id, num_jobs)
    index_adjust=start_job
    print(f'start_job = {start_job}, end_job = {end_job}')

start_job = 18, end_job = 20


In [83]:
data=data.isel(time=slice(start_job,end_job))

In [93]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/job_out/'
open_file=dir2+f'LFC_LCL_binary_array_{res}_{t_res}_{Np_str}_{job_id}.h5'
with h5py.File(open_file, 'r') as f:
    # Load the dataset by its name
    LFC = f['LFC'][:]
    print(LFC.shape)


(2, 50653000)


In [85]:
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'
with h5py.File(in_file, 'r') as f:
    # Load the dataset by its name
    W = f['W'][start_job:end_job]
    QCQI = f['QCQI'][start_job:end_job]
    A_c = f['A_c'][start_job:end_job]
    Z = f['Z'][start_job:end_job]
    Y = f['Y'][start_job:end_job]
    X = f['X'][start_job:end_job]
    parcel_z=f['z'][start_job:end_job]

LFC_data=data['lfc'].data

In [ ]:
#TESTING IF RESULT IS CORRECT

    
for p in range(1000):
    
    t=np.arange(len(data['time']))
    z=parcel_z[:,p]
    y=Y[:,p]
    x=X[:,p]
    LFC_1=LFC_data[t,y,x]
    
    LFC_2=LFC[:,p]
    
    sub=z-LFC_1; 
    sub2 = np.where(sub > 0, True, False)
    
    # print(sub[0:15])
    # print(sub2[0:15])
    # print(LFC_2[0:15])

    if np.any(sub2!=LFC_2):
        print(f'p={p}')

In [89]:
#TESTING W
for p in range(100):
    t=np.arange(len(data['time']))
    z=Z[:,p]
    y=Y[:,p]
    x=X[:,p]
    
    W1 = data['winterp'].isel(time=xr.DataArray(t, dims='dim'),
                              zh=xr.DataArray(z, dims='dim'),
                              yh=xr.DataArray(y, dims='dim'),
                              xh=xr.DataArray(x, dims='dim')).data
    W2=W[:,p]

    if np.any(W1!=W2):
        print(f'p={p}')

In [90]:
#TESTING QCQI
    
for p in range(100):
    t=np.arange(len(data['time']))
    z=Z[:,p]
    y=Y[:,p]
    x=X[:,p]
    
    QC = data['qc'].isel(time=xr.DataArray(t, dims='dim'),
                              zh=xr.DataArray(z, dims='dim'),
                              yh=xr.DataArray(y, dims='dim'),
                              xh=xr.DataArray(x, dims='dim')).data
    QI = data['qi'].isel(time=xr.DataArray(t, dims='dim'),
                              zh=xr.DataArray(z, dims='dim'),
                              yh=xr.DataArray(y, dims='dim'),
                              xh=xr.DataArray(x, dims='dim')).data
    QCQI1=QC+QI
    QCQI2=QCQI[:,p]

    if np.any(W1!=W2):
        print(f'p={p}')
    if np.any(QCQI1!=QCQI2):
        print(f'p={p}')

In [35]:
#TESTING A_C
#TESTING W
for p in range(100):
    t=np.arange(len(data['time']))
    z=Z[:,p]
    y=Y[:,p]
    x=X[:,p]
    
    W1 = data['winterp'].isel(time=xr.DataArray(t, dims='dim'),
                              zh=xr.DataArray(z, dims='dim'),
                              yh=xr.DataArray(y, dims='dim'),
                              xh=xr.DataArray(x, dims='dim')).data
    W2=W[:,p]

    QC = data['qc'].isel(time=xr.DataArray(t, dims='dim'),
                              zh=xr.DataArray(z, dims='dim'),
                              yh=xr.DataArray(y, dims='dim'),
                              xh=xr.DataArray(x, dims='dim')).data
    QI = data['qi'].isel(time=xr.DataArray(t, dims='dim'),
                              zh=xr.DataArray(z, dims='dim'),
                              yh=xr.DataArray(y, dims='dim'),
                              xh=xr.DataArray(x, dims='dim')).data
    QCQI1=QC+QI
    QCQI2=QCQI[:,p]

    A_c1=A_c[:,p]
    cond1=W1>=0.5;cond2=QCQI1>=1e-6
    A_c2=np.where(cond1&cond2,1,0)

    if np.any(A_c1!=A_c2):
        print(f'p={p}')